<a href="https://colab.research.google.com/github/SahazaFitia-glitch/mon-projet-colab/blob/main/ANDRIAMANANA_Hery_Sahaza_Fitia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
#  RECONNAISSANCE DE VOIX : "C'est moi" / "C'est lui" / "Autre"
#  Une seule cellule. Seule la section 0 est à modifier.
# ============================================================

# ===== 0. RÉGLAGES (à modifier) =====
DOSSIER_REF  = "/home/ubuntu/Entraînement/Modele"  # contient les dossiers Moi, Lui, Autre
AUDIO_URL    = "/home/ubuntu/Entraînement/Test/Autre/Autre8.wav"          # lien OU chemin d'un fichier local
RECONSTRUIRE = False                               # True si vous avez ajouté/changé des fichiers
SEUIL_MANUEL = None                                # ex. 0.5 pour forcer le seuil

# ===== 1. Installation =====
!pip install -q speechbrain gdown librosa soundfile requests

# ===== 2. Imports =====
import os, re, glob, warnings, urllib.parse
import numpy as np, requests, librosa, torch, gdown
warnings.filterwarnings("ignore")
try:
    from speechbrain.inference.speaker import EncoderClassifier
except ImportError:  # anciennes versions de SpeechBrain
    from speechbrain.pretrained import EncoderClassifier

SR = 16000                                   # fréquence audio utilisée partout
EXTENSIONS = (".wav", ".mp3", ".m4a", ".flac", ".ogg")
DOSSIERS = {"MOI": "Moi", "LUI": "Lui", "AUTRE": "Autre"}   # classe -> nom du dossier
DUREE_SEG = 3.0                              # l'audio est découpé en morceaux de 3 s
DUREE_MIN = 1.5                              # durée minimale utile (en secondes)

# ===== 3. Modèle préentraîné de reconnaissance du locuteur (ECAPA-TDNN) =====
# Il transforme une voix en "empreinte" de 192 nombres. Chargé une seule fois par session.
device = "cuda" if torch.cuda.is_available() else "cpu"
if "classifieur" not in globals():
    classifieur = EncoderClassifier.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="/content/modele_ecapa",
        run_opts={"device": device})

# ===== 4. Fonctions audio =====
def charger_audio(chemin):
    """Charge, nettoie et normalise un audio. Lève une erreur s'il est vide ou trop court."""
    y, _ = librosa.load(chemin, sr=SR, mono=True)
    if y.size == 0:
        raise ValueError("Fichier audio vide ou illisible.")
    y, _ = librosa.effects.trim(y, top_db=30)            # enlève les silences aux extrémités
    if y.size < DUREE_MIN * SR:
        raise ValueError(f"Audio trop court ({y.size / SR:.1f} s, minimum {DUREE_MIN} s).")
    m = np.max(np.abs(y))
    return y / m if m > 0 else y                          # normalise le volume

def decouper(y):
    n = int(DUREE_SEG * SR)
    segs = [y[i:i + n] for i in range(0, len(y) - n + 1, n)]
    return segs if segs else [y]

def embedding(seg):
    with torch.no_grad():
        e = classifieur.encode_batch(torch.tensor(seg).float().unsqueeze(0).to(device))
    e = e.squeeze().cpu().numpy()
    return e / np.linalg.norm(e)

def embeddings_audio(y):
    """Retourne les empreintes de chaque morceau + l'empreinte moyenne de l'audio."""
    E = np.array([embedding(s) for s in decouper(y)])
    moy = E.mean(axis=0)
    return E, moy / np.linalg.norm(moy)

# ===== 5. Fonctions de décision =====
# LIMITES : avec peu d'enregistrements, le résultat est fragile.
#  - MOI et LUI sont représentés par la moyenne de leurs fichiers.
#  - AUTRE est comparé à l'exemple le plus proche (quelques personnes ne représentent pas "tout le monde").
#  - Un audio bruité, un autre micro ou une voix inconnue très proche de MOI/LUI peut être mal classé.
#  - Le seuil est estimé automatiquement sur VOS fichiers, il reste approximatif.
def construire(refs):
    modele = {}
    for c in ("MOI", "LUI"):
        p = np.array([r["moy"] for r in refs if r["classe"] == c]).mean(axis=0)
        modele[c] = p / np.linalg.norm(p)
    A = [r["E"] for r in refs if r["classe"] == "AUTRE"]
    modele["AUTRE"] = np.vstack(A) if A else None
    return modele

def scores(emb, modele):
    s = {c: float(emb @ modele[c]) for c in ("MOI", "LUI")}
    s["AUTRE"] = float(np.max(modele["AUTRE"] @ emb)) if modele["AUTRE"] is not None else -1.0
    return s

def decider(emb, modele, seuil):
    s = scores(emb, modele)
    meilleur = max(("MOI", "LUI"), key=lambda c: s[c])
    if s[meilleur] < seuil or s["AUTRE"] > s[meilleur]:   # incertain, ou plus proche d'AUTRE
        return "AUTRE", s
    return meilleur, s

# ===== 6. Lecture des dossiers et construction du système =====
def preparer():
    if not os.path.isdir(DOSSIER_REF):
        raise FileNotFoundError(f"Dossier introuvable : {DOSSIER_REF}")
    refs, comptes = [], {}
    for classe, nom_dossier in DOSSIERS.items():
        fichiers = sorted(f for f in glob.glob(os.path.join(DOSSIER_REF, nom_dossier, "*"))
                          if f.lower().endswith(EXTENSIONS))
        comptes[classe] = len(fichiers)
        for f in fichiers:
            try:
                E, moy = embeddings_audio(charger_audio(f))
                refs.append({"classe": classe, "fichier": os.path.basename(f), "E": E, "moy": moy})
            except Exception as e:
                print("Fichier ignoré :", os.path.basename(f), "->", e)
    for c, n in comptes.items():
        print(f"Nombre de fichiers {c} : {n}")
    nb = {c: sum(r["classe"] == c for r in refs) for c in ("MOI", "LUI")}
    for c in ("MOI", "LUI"):
        if nb[c] == 0:
            raise ValueError(f"Aucun fichier {c} valide dans {os.path.join(DOSSIER_REF, DOSSIERS[c])}.")
    if comptes["AUTRE"] == 0:
        print("ATTENTION : dossier Autre vide, la catégorie Autre reposera uniquement sur le seuil.")

    # Calibrage du seuil et évaluation "leave-one-out" :
    # chaque fichier est testé avec un modèle construit SANS lui.
    if min(nb.values()) >= 2:
        vrais, faux = [], []
        for r in refs:
            m = construire([x for x in refs if x is not r])
            if r["classe"] in ("MOI", "LUI"):
                autre = "LUI" if r["classe"] == "MOI" else "MOI"
                vrais.append(float(r["moy"] @ m[r["classe"]]))
                faux.append(float(r["moy"] @ m[autre]))
            else:
                faux += [float(r["moy"] @ m["MOI"]), float(r["moy"] @ m["LUI"])]
        seuil = (np.mean(vrais) + np.mean(faux)) / 2
        print(f"Similarité moyenne (même voix) : {np.mean(vrais):.2f} | (autre voix) : {np.mean(faux):.2f}")
    else:
        seuil = 0.45
        print("Moins de 2 fichiers pour MOI ou LUI : seuil par défaut 0.45 (peu fiable).")
    if SEUIL_MANUEL is not None:
        seuil = SEUIL_MANUEL
    print(f"Seuil utilisé : {seuil:.2f}")

    if min(nb.values()) >= 2:
        bons = 0
        for r in refs:
            pred, _ = decider(r["moy"], construire([x for x in refs if x is not r]), seuil)
            bons += (pred == r["classe"])
            if pred != r["classe"]:
                print(f"  erreur : {r['fichier']} ({r['classe']}) -> {pred}")
        print(f"Précision leave-one-out : {bons}/{len(refs)} = {bons / len(refs):.0%}")
    return construire(refs), seuil

# La préparation n'est refaite que si nécessaire (sinon relancer la cellule est rapide)
if RECONSTRUIRE or "MODELE" not in globals():
    MODELE, SEUIL = preparer()

# ===== 7. Téléchargement de l'audio à tester =====
# ACCEPTÉS : chemin d'un fichier local, lien direct vers un fichier audio (.wav, .mp3, .m4a...),
#            lien de partage Google Drive ("Toute personne disposant du lien"), lien Dropbox.
# NON ACCEPTÉS : YouTube, Spotify, SoundCloud et pages web (pas de fichier direct).
def telecharger_audio(url, dest="/content/audio_test"):
    url = url.strip()
    if os.path.isfile(url):
        return url
    if not url.startswith(("http://", "https://")):
        raise ValueError("Entrez un lien commençant par http(s):// ou le chemin d'un fichier existant.")
    if any(s in url for s in ("youtube.com", "youtu.be", "spotify.com", "soundcloud.com")):
        raise ValueError("Lien de streaming non accepté : utilisez un lien direct vers un fichier audio.")
    if "drive.google.com" in url or "docs.google.com" in url:
        m = re.search(r"/d/([\w-]+)", url) or re.search(r"[?&]id=([\w-]+)", url)
        if not m:
            raise ValueError("Lien Google Drive non reconnu.")
        sortie = gdown.download(id=m.group(1), output=dest + "_drive", quiet=True)
        if not sortie:
            raise ConnectionError("Téléchargement Drive impossible : partage requis 'Toute personne disposant du lien'.")
        return sortie
    if "dropbox.com" in url:
        url = url.replace("dl=0", "dl=1")
    try:
        r = requests.get(url, stream=True, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        raise ConnectionError(f"Lien inaccessible ou problème de connexion : {e}")
    type_ = r.headers.get("Content-Type", "").lower()
    if "text/html" in type_:
        raise ValueError("Ce lien mène à une page web, pas à un fichier audio direct.")
    ext = os.path.splitext(urllib.parse.urlparse(url).path)[1].lower()
    if ext not in EXTENSIONS:
        ext = ".mp3" if "mpeg" in type_ else ".wav"
    chemin = dest + ext
    with open(chemin, "wb") as f:
        for bloc in r.iter_content(8192):
            f.write(bloc)
    if os.path.getsize(chemin) == 0:
        raise ValueError("Le fichier téléchargé est vide.")
    return chemin

# ===== 8. Prédiction et résultat final =====
print("\n" + "=" * 40)
try:
    chemin = telecharger_audio(AUDIO_URL)
    _, emb = embeddings_audio(charger_audio(chemin))
    classe, s = decider(emb, MODELE, SEUIL)
    print("RÉSULTAT :", {"MOI": "C'est moi", "LUI": "C'est lui", "AUTRE": "Autre"}[classe])
    print(f"Similarités -> moi : {s['MOI']:.2f} | lui : {s['LUI']:.2f} | "
          f"autre : {s['AUTRE']:.2f} | seuil : {SEUIL:.2f}")
except (ValueError, ConnectionError) as e:
    print("Erreur :", e)
except Exception as e:
    print("Format audio non pris en charge ou erreur inattendue :", e)


RÉSULTAT : Autre
Similarités -> moi : 0.42 | lui : 0.28 | autre : 0.56 | seuil : 0.39


In [ ]:
from google.colab import drive
drive.mount('/content/drive')